<a href="https://colab.research.google.com/github/GretelKMendez/Tareas-Mac-IA/blob/main/SpotifyGK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# 1. Cargar el DataFrame
# Usamos el dataset de Spotify (asegúrate de tener el archivo en la misma carpeta)
df = pd.read_csv('SpotifyFeatures.csv')

# INSPECCIÓN INICIAL

print("--- Primeras 5 filas del DataFrame ---")
print(df.head())

print("\n--- Información general y tipos de datos ---")
print(df.info())

print("\n--- Conteo de valores nulos por columna ---")
print(df.isnull().sum())

# PROCESO DE LIMPIEZA DE DATOS

# 2. Eliminación de duplicados
# A veces una misma canción aparece varias veces si está en diferentes álbumes
df = df.drop_duplicates()

# 3. Manejo de valores nulos (si los hubiera)
# Opción A: Eliminar filas con cualquier valor nulo
df = df.dropna()

# 4. Transformación de tipos (Data Casting)
# Asegurarnos de que la popularidad sea un número entero
df['popularity'] = df['popularity'].astype(int)

# 5. Filtrado de columnas irrelevantes
# Para el análisis de audio, no necesitamos IDs largos ni nombres que no aportan patrones
columnas_a_eliminar = ['track_id', 'track_name', 'artist_name']
df_limpio = df.drop(columns=columnas_a_eliminar)

# 6. Manejo de Outliers (Valores atípicos)
# Por ejemplo, si hay canciones con duración 0 o negativa (errores de carga)
df_limpio = df_limpio[df_limpio['duration_ms'] > 0]

# 7. Resetear el índice
# Después de borrar filas, los números de las filas quedan saltados. Esto los arregla.
df_limpio = df_limpio.reset_index(drop=True)

print("\n--- DataFrame Final después de la limpieza ---")
print(df_limpio.head())
print(f"\nDimensiones finales: {df_limpio.shape}")

--- Primeras 5 filas del DataFrame ---
   genre        artist_name                        track_name  \
0  Movie     Henri Salvador       C'est beau de faire un Show   
1  Movie  Martin & les fées  Perdu d'avance (par Gad Elmaleh)   
2  Movie    Joseph Williams    Don't Let Me Be Lonely Tonight   
3  Movie     Henri Salvador    Dis-moi Monsieur Gordon Cooper   
4  Movie       Fabien Nataf                         Ouverture   

                 track_id  popularity  acousticness  danceability  \
0  0BRjO6ga9RKCKjfDqeFgWV           0         0.611         0.389   
1  0BjC1NfoEOOusryehmNudP           1         0.246         0.590   
2  0CoSDzoNIKCRs124s9uTVy           3         0.952         0.663   
3  0Gc6TVm52BwZD07Ki6tIvf           0         0.703         0.240   
4  0IuslXpMROHdEPvSl1fTQK           4         0.950         0.331   

   duration_ms  energy  instrumentalness key  liveness  loudness   mode  \
0        99373   0.910             0.000  C#    0.3460    -1.828  Major   
1    

In [6]:

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# PASO 1: Carga de datos
# Cargamos el dataset descargado de Kaggle
df = pd.read_csv('SpotifyFeatures.csv')

# PASO 2: Selección y Limpieza
# Elegimos solo variables numéricas que describen el audio
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness',
            'valence', 'tempo', 'duration_ms']

X = df[features]
y = df['popularity'] # Nuestra variable objetivo

# PASO 3: División del dataset
# Separamos 80% para entrenar y 20% para evaluar si el modelo aprendió
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# PASO 4: Escalado (Fundamental para Lasso)
# Lasso es sensible a las escalas (ej. el tempo llega a 200 y la energía a 1.0).
# El StandardScaler pone todo en una misma escala (media 0, varianza 1).
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# PASO 5: Modelo 1 - Regresión Lasso
# alpha es la fuerza de la penalización. Un alpha mayor elimina más variables.
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)

# --- Justo después de y_pred_lasso = lasso.predict(X_test_scaled) ---

print("--- RESULTADOS REGRESIÓN LASSO ---")
# Error Cuadrático Medio (mientras más bajo, mejor)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
# R2 (mientras más cercano a 1, mejor)
r2_lasso = r2_score(y_test, y_pred_lasso)

print(f"Error Cuadrático Medio (MSE): {mse_lasso:.2f}")
print(f"Coeficiente de determinación (R2): {r2_lasso:.2f}")
comparativa_rf = pd.DataFrame({
    'Valor Real': y_test.values,
    'Predicción RF': y_pred_rf
})

print("\n--- COMPARATIVA REAL VS RF (Primeras 5 filas) ---")
print(comparativa_rf.head())

# PASO 6: Modelo 2 - Bosques Aleatorios (Regresión)
# Usamos 10 árboles para promediar las predicciones y que sea mas rapida la lectura.
rf = RandomForestRegressor(n_estimators=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train) # RF no necesita obligatoriamente datos escalados
y_pred_rf = rf.predict(X_test)

comparativa_rf = pd.DataFrame({
    'Valor Real': y_test.values,
    'Predicción RF': y_pred_rf
})

print("\n--- COMPARATIVA REAL VS RF (Primeras 5 filas) ---")
print(comparativa_rf.head())

# --- Justo después de y_pred_rf = rf.predict(X_test) ---

print("--- RESULTADOS BOSQUES ALEATORIOS ---")
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Error Cuadrático Medio (MSE): {mse_rf:.2f}")
print(f"Coeficiente de determinación (R2): {r2_rf:.2f}")


--- RESULTADOS REGRESIÓN LASSO ---
Error Cuadrático Medio (MSE): 255.45
Coeficiente de determinación (R2): 0.23

--- COMPARATIVA REAL VS RF (Primeras 5 filas) ---
   Valor Real  Predicción RF
0          45           51.8
1          25           28.5
2          19           47.7
3          29           43.7
4          17           22.8

--- COMPARATIVA REAL VS RF (Primeras 5 filas) ---
   Valor Real  Predicción RF
0          45           51.8
1          25           28.5
2          19           47.7
3          29           43.7
4          17           22.8
--- RESULTADOS BOSQUES ALEATORIOS ---
Error Cuadrático Medio (MSE): 155.66
Coeficiente de determinación (R2): 0.53


In [7]:
# Definimos los datos de una nueva canción (puedes inventarlos o sacarlos de la API de Spotify)
nueva_cancion = pd.DataFrame([{
    'danceability': 0.75,
    'energy': 0.8,
    'loudness': -5.0,
    'speechiness': 0.05,
    'acousticness': 0.1,
    'instrumentalness': 0.0,
    'liveness': 0.12,
    'valence': 0.6,
    'tempo': 120.0,
    'duration_ms': 210000
}])

In [8]:
# 1. Escalamos los datos de la nueva canción
nueva_cancion_scaled = scaler.transform(nueva_cancion)

# 2. Predecimos el puntaje de popularidad (0-100)
prediccion_lasso = lasso.predict(nueva_cancion_scaled)

print(f"Puntaje de popularidad estimado (Lasso): {prediccion_lasso[0]:.2f}")

Puntaje de popularidad estimado (Lasso): 49.92


In [10]:
# Predecimos la categoría (1 = Popular, 0 = No popular)
prediccion_rf = rf.predict(nueva_cancion)

if prediccion_rf[0] == 1:
    print("¡El modelo predice que la canción será un ÉXITO!")
else:
    print("El modelo predice que la canción NO será popular.")

El modelo predice que la canción NO será popular.
